# Halo Power Spectra for Dark Photon-Galaxy Cross-Correlations

## Setup


In [ ]:
import os, sys
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
from matplotlib.lines import Line2D


%load_ext autoreload
%autoreload 2
%matplotlib inline

from plot_params import params

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
from matplotlib import gridspec
from matplotlib import ticker

from scipy.interpolate import interp1d
import numpy as np

import pymaster as nmt
# Cosmology and astrophysics
import astropy.units as un
from astropy.cosmology import Planck18  # Standard Planck 2018 cosmology
import astropy.constants as const
import numpy as np

# Numerical integration and interpolation
from scipy.integrate import simpson
from scipy.interpolate import interp1d
import scipy.special as sp


from tqdm import tqdm

# Halo modeling packages
import halomod as hd  # Main halo model framework
import hmf           # Halo mass function calculations

pylab.rcParams.update(params)

cols_default = plt.rcParams['axes.prop_cycle'].by_key()['color']


## 2. Halo Model Configuration

Set up the halo model framework including:
- Custom concentration-mass relation (Bhattacharya et al. 2012)
- Dark photon coupling parameters
- Cosmological and numerical grids

In [ ]:
# need to define concentration
class Bhattacharya12(hd.concentration.CMRelation):
    _defaults = {"a": None, "b": None, "c": None}
    native_mdefs = (hmf.mass_definitions.SOCritical(), hmf.mass_definitions.SOVirial())

    def cm(self, m, z):
        set_params = {
            "200c": {
                "a": 0.54,
                "b": 5.9,
                "c": -0.35,
                },
            "178c": {
                "a": 0.9,
                "b": 7.7,
                "c": -0.29,
                },
            "vir": {
                "a": 0.9,
                "b": 7.7,
                "c": -0.29,
                }
        }
        parameter_set = set_params.get(self.mdef.colossus_name, set_params["200c"])
        nu = self.growth.growth_factor(z)**-1 * (1.12 * (m/5e13)**0.3 + 0.53)
        return self.growth.growth_factor(z)**parameter_set["a"] * parameter_set["b"] * nu**parameter_set["c"]
    
kappa = 5.7e-38 # eV**2

# arrays of redshift, halo masses, and wavenumbers
zs = np.linspace(0.005, 4, 100) 
ms = np.geomspace(1e11, 1e17, 100,) #Msolar
# ms = np.geomspace(7e8, 3.5e15, 100)/Planck18.h #Msun
dlogm = np.diff(np.log10(ms*Planck18.h))[0]
ks = np.geomspace(1e-4, 1e3, 10**4,) # Mpc**-1
dlnk = np.diff(np.log(ks/Planck18.h))[0]

hm = hd.DMHaloModel(
    z=0.0,
    Mmin=np.log10(ms[0]*Planck18.h), #Msun/h
    Mmax=np.log10(ms[-1]*Planck18.h),#+ dlogm, #Msun/h
    dlog10m=dlogm,
    lnk_min=np.log(ks[0]/Planck18.h), #h/Mpc
    lnk_max=np.log(ks[-1]/Planck18.h) + dlnk, #h / Mpc
    dlnk=dlnk,
    halo_concentration_model="Bhattacharya12",
    bias_model="Tinker10",
    # halo_overdensity_crit=200,
    # disable_mass_conversion=True,
    hmf_model="Tinker08",
    mdef_model=hmf.halos.mass_definitions.SOVirial,
    # mdef_params={"overdensity":200},
    cosmo_params={"m_nu":[0,0,0], },
    )

rhocritz = Planck18.critical_density(zs).to(un.Msun / un.Mpc**3).value # Msun / Mpc**3

def rho_gas(r, z, m200, R200):
    x = r / R200
    p = {"rho0": {
            "a": 4e3,
            "b": 0.29,
            "c": -0.66,
        },
         "alpha": {
            "a": 0.88,
            "b": -0.03,
            "c": 0.19,
         },
         "beta": {
            "a": 3.83,
            "b": 0.04,
            "c": -0.025,
         }
         }
    gamma = -0.2
    xc = 0.5
    rho0  = p["rho0"]["a"]  * (m200[..., None] / 10**14)**p["rho0"]["b"]  * (1+z[:, None, None])**p["rho0"]["c"]
    alpha = p["alpha"]["a"] * (m200[..., None] / 10**14)**p["alpha"]["b"] * (1+z[:, None, None])**p["alpha"]["c"]
    beta  = p["beta"]["a"]  * (m200[..., None] / 10**14)**p["beta"]["b"]  * (1+z[:, None, None])**p["beta"]["c"]
    return Planck18.Ob0 / Planck18.Om0 * rhocritz[:, None, None] * rho0 * (x /xc)**gamma * (1 + (x/xc)**alpha)**(-(beta+gamma)/alpha)


## 3. Compute Halo Properties

Calculate halo properties across the redshift-mass grid:
- Bias factors
- Number densities  
- Concentration parameters
- Virial and R200 radii
- Linear matter power spectrum

This step is computationally intensive as it involves solving for halo properties at each (z,M) point.

In [ ]:
# ============================================================================
# Initialize Arrays for Halo Properties
# ============================================================================


bias      = np.empty((len(zs), len(ms))) # Halo bias factors (dimensionless)
n         = np.empty((len(zs), len(ms))) # Number density dn/dM (Mpc^-3 Msun^-1) 
cs        = np.empty((len(zs), len(ms))) # Concentration parameters (dimensionless)
rvirs     = np.empty((len(zs), len(ms))) # Virial radii (Mpc)
R200s     = np.empty((len(zs), len(ms))) # R200 radii (Mpc)
m200s     = np.empty((len(zs), len(ms))) # M200 masses - mass within R200 (Msun)
c200s     = np.empty((len(zs), len(ms))) # Concentration for M200 definition (dimensionless)
lin_power = np.empty((len(zs), len(ks))) # Linear matter power spectrum P(k,z) (Mpc^3)

# ============================================================================
# Main Computation Loop
# ============================================================================

for i, z in tqdm(enumerate(zs), total=len(zs), desc="Computing halo properties"):
    # Update halo model to current redshift
    hm.z = z    

    # Extract halo properties at this redshift (vectorized over mass)
    n[i, :] = hm.dndm * Planck18.h**4          # Convert to physical units
    bias[i, :] = hm.halo_bias                  # Already dimensionless
    cs[i, :] = hm.cmz_relation                 # Already dimensionless  
    
    # Linear power spectrum (convert from h units to physical)
    lin_power[i, :] = hm.linear_power_fnc(ks / Planck18.h) / Planck18.h**3
    
    # Mass-dependent properties (requires loop over masses)
    for j, m in enumerate(ms):
        # Virial radius from halo profile (convert from h units)
        rvirs[i, j] = hm.halo_profile.halo_mass_to_radius(
            m * Planck18.h, at_z=z
        ) / Planck18.h
        
        # Convert from virial to M200c definition
        # This gives M200, R200, and c200 for the 200×critical density definition
        m200s[i, j], R200s[i, j], c200s[i, j] = hm.mdef.change_definition(
            m * Planck18.h,  # Input mass in h units
            mdef=hmf.halos.mass_definitions.SOCritical(overdensity=200),
            c=cs[i, j],      # Input concentration
            z=z,
            cosmo=Planck18
        )

# Convert masses and radii from h units to physical units
m200s /= Planck18.h  # Msun
R200s /= Planck18.h  # Mpc

In [ ]:
omega = 2* np.pi * (6.58212e-7 * (1+zs)) # eV
rhoc = 3.62158e-11 #eV^4
fg = rhoc * (6e-8) * (1+zs)**(-5/37) * np.pi**2 / (omega**4)
omegap = 1e-13
chis = Planck18.comoving_distance(zs).to(un.Mpc).value
K12 = 5.7e-37 # eV^-1
D = 10e3 # kpc
Deltal = 50
kappa = 5.81132e-28 #eV^-1
B = 0.0195353 * (0.1e-6) # eV^2
P = D * (kappa * B)**2 / (Deltal) * (16 * omega**4 - omegap**4) / (16 * omega**2 * omegap**4)
Ptot = simpson(simpson(fg * P[:, None] * chis[:, None]**2 * n, x=chis, axis=0), x=ms)
print("Ptot = ", Ptot)

## 4. Galaxy Modeling and Angular Power Spectra

Set up galaxy catalogs using Halo Occupation Distribution (HOD) models and compute:
- Galaxy number densities and selection functions
- Galaxy auto-correlation angular power spectra (1-halo + 2-halo terms)
- Limber approximation for efficient angular power spectrum calculation

We use parameters from Kusiak et al. 2022/2023.

In [ ]:
# ============================================================================
# Comoving Distances and Power Spectrum Interpolation  
# ============================================================================

# Comoving distances for Limber approximation (Mpc)
chis = Planck18.comoving_distance(zs).to(un.Mpc).value

# Cross-redshift power spectrum matrix for 2-halo term
# P_lin(k, z1, z2) = sqrt(P_lin(k, z1) * P_lin(k, z2))
Plin_z1_z2 = np.sqrt(lin_power[None, :] * lin_power[:, None])
interpolated_Plin = interp1d(ks, lin_power, axis=-1, bounds_error=False, fill_value=0)

print(f"Comoving distance range: {chis.min():.1f} - {chis.max():.1f} Mpc")

# ============================================================================
# Galaxy Number Density and Selection Function
# ============================================================================

# Load galaxy redshift distribution for unWise
dNgdz_dat = np.genfromtxt("data/galaxy_dNg_dz.csv", delimiter=",").T
dNgdz = interp1d(dNgdz_dat[0], dNgdz_dat[1], bounds_error=False, fill_value=0.0)

# Angular multipoles for power spectrum calculation
ls = np.geomspace(100, 4000, 20, dtype=int)

# ============================================================================
# Halo Occupation Distribution (HOD) Models
# ============================================================================

# HOD parameters from Kusiak et al. 2022 (K22) and 2023 (K23)
# These describe how galaxies populate dark matter halos
HOD_model = {
    "K22": {
        "logm_min": 11.97,    # log10 Msun
        "sigma_logm": 0.687,    
        "alpha_s": 1.304,       
        "m1prime": 10**12.87, # Msun  
        "lambdaNFW": 1.087,     
    },
    "K23": {
        "logm_min": 11.86,    # log10 Msun
        "sigma_logm": 0.020,    
        "alpha_s": 1.06,       
        "m1prime": 10**12.78, # Msun
        "lambdaNFW": 1.80,   
    }
}

# Select which HOD model to use
model = "K22"  # Change to "K23" for the updated model
print(f"Using HOD model: {model}")

# ============================================================================ 
# HOD Functions
# ============================================================================

def Nc(m):
    logm_min = HOD_model[model]["logm_min"]
    sigma_logm = HOD_model[model]["sigma_logm"]
    return 0.5 * (1 + sp.erf((np.log10(m) - logm_min) / sigma_logm))

def Ns(m):
    alpha_s = HOD_model[model]["alpha_s"]
    m1prime = HOD_model[model]["m1prime"]
    return Nc(m) * (m / m1prime)**alpha_s

# Mean galaxy number density per redshift slice (galaxies/Mpc³) 
ngbar = simpson(n * (Nc(ms)[None, :] + Ns(ms)[None, :]), x=ms, axis=1)


def W(z):
    return ((Planck18.H(z) / const.c) / Planck18.comoving_distance(z)**2).to(un.Mpc**-3).value * dNgdz(z)

def ug(Ws, Ncs, Nss, l):
    return Ws[:, None] * ngbar[:, None]**-1 * (Ncs[None, :] + Nss[None, :] * um(l))

def um(l):
    """
    Fourier transform of NFW satellite distribution.
    """
    lambdaNFW = HOD_model[model]["lambdaNFW"]
    
    # Convert angular scale to physical wavevector  
    ks_angular = (l + 0.5) / chis[:, None]  # k = (l+0.5)/chi in Limber approximation
    q = ks_angular * R200s / c200s * (1 + zs[:, None])  
    
    x = lambdaNFW * c200s  # Satellite concentration parameter
    qtilde = (1 + x) * q  
    
    # Sine and cosine integrals for NFW Fourier transform
    Si_q, Ci_q = sp.sici(q)
    Si_qtilde, Ci_qtilde = sp.sici(qtilde)
    
    # NFW normalization factor
    fNFW = (np.log(1 + x) - x / (1 + x))**-1
    
    # Full NFW Fourier transform
    return ((np.cos(q) * (Ci_qtilde - Ci_q) + 
             np.sin(q) * (Si_qtilde - Si_q) - 
             np.sin(x * q) / qtilde) * fNFW)

def ug_mom2(Ws, Nss, l):
    """
    Second moment of galaxy profile for 1-halo term calculation.
    """
    um_l = um(l)
    return Ws[:, None]**2 * ngbar[:, None]**-2 * (Nss[None, :]**2 * um_l**2 + 2 * Nss * um_l)

# ============================================================================
# Precompute Arrays for Efficiency
# ============================================================================

# Pre-calculate arrays that don't depend on multipole l
Ncs = Nc(ms)
Nss = Ns(ms)  
Ws = W(zs)

def Cl_1h_gg(l):
    return np.trapz(chis**2 / ((Planck18.H(zs) / const.c).to(un.Mpc**-1).value) * np.trapz(n * ug_mom2(Ws, Nss, l=l), x=ms, axis=1), x=zs, axis=0)

interp_lin_power = interp1d(ks, lin_power, axis=-1)

def Cl_2h_gg_limber(l):
    m_int = simpson(n * bias * ug(Ws, Ncs, Nss, l=l), x=ms, axis=1)
    Plin_at_zs = np.array([interp_lin_power((l+0.5)/chi)[chis == chi] for chi in chis]).reshape(zs.shape)
    return simpson(chis**2 / ((Planck18.H(zs) / const.c).to(un.Mpc**-1).value) * m_int**2 * Plin_at_zs, x=zs, axis=0)

def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx], idx

# ============================================================================
# Compute Galaxy Auto-Power Spectrum
# ============================================================================

print("Computing galaxy auto-power spectrum...")
print("This calculation includes both 1-halo and 2-halo terms...")

Cl_gg = np.array([Cl_1h_gg(l) + Cl_2h_gg_limber(l) for l in tqdm(ls, desc="Galaxy C_l")])

print(f"Galaxy auto-power spectrum computed!")

## 5. Dark Photon Conversion Modeling

Set up the dark photon to photon conversion calculation:
- Compute 3D gas density profiles around halos
- Calculate effective photon mass $m_\gamma = \sqrt{\kappa \rho_{gas}}$ 
- Find conversion radii where $m_\gamma = m_A$ (resonant conversion)
- Compute conversion probability and angular correlations


In [ ]:
# compute plasma mass inside each halo
rs = np.linspace(1e-12, 1.1 * rvirs, int(1e4))  # Shape: (nz, nm, nr)
rs = np.moveaxis(rs, 0, -1)  # Move radius axis to end for broadcasting
rho_gases = rho_gas(rs, zs, m200s, R200s[..., None])

mgamma2 = kappa * rho_gases  # Units: eV²

bessel_arg1 = ks[None, None, :] * chis[:, None, None]  # k * chi(z1)
bessel_arg2 = ks[None, None, :] * chis[None, :, None]  # k * chi(z2)

# Power spectrum integrand (before Bessel function convolution)
integrand_without_js = Plin_z1_z2 * ks[None, None, :] ** 2

def hl(l):
    return (
        (2 * l + 1) / (2 * np.pi**2) *
        simpson(
            integrand_without_js *
            sp.spherical_jn(l, bessel_arg1) *
            sp.spherical_jn(l, bessel_arg2),
            x=ks,
            axis=-1,
        )
    )

# Compute h_l for all multipoles (computationally intensive)
hls = np.array([hl(l) for l in tqdm(ls, desc="Computing h_l")])

In [ ]:
# ============================================================================
# Main Cross-Correlation Calculation Function
# ============================================================================

# here we compute the dark photon auto correlation and cross correlation in the halo model
def get_Cls(mA):
    # Initialize arrays for conversion properties
    drho_drs = np.empty([len(zs), len(ms)])    # Gas density gradient at conversion
    r_convs = np.empty([len(zs), len(ms)])     # Conversion radius
        
    # Loop over redshift and mass to find where m_γ = m_A
    for i in range(len(zs)):
        for j in range(len(ms)):
            # Find radius where effective photon mass equals dark photon mass
            mgamma_profile = np.sqrt(mgamma2[i, j, :])
            
            # Find nearest point to target mass
            mA_value, mA_index = find_nearest(mgamma_profile, mA)
            
            # Compute gas density gradient for conversion probability
            drho_dr = np.abs(np.gradient(rho_gases[i, j, :], rs[i, j, :]))
            drho_drs[i, j] = np.abs(drho_dr[mA_index])
            
            # Store conversion radius
            r_convs[i, j] = rs[i, j, mA_index]
                    

    drho_drs_converted = drho_drs / 1.567e29  # Convert to appropriate units
    
    theta_max_array = r_convs * (1 + zs[:, None]) / chis[:, None]
    
    # Dark photon coupling parameters
    epsilon = 1.0      # Dark photon-photon mixing parameter (set to 1 for now)
    omega0  = 1.0       # Characteristic frequency in eV
    
    # Overall constant for conversion probability
    const_prefact = epsilon**2 * mA**4 / omega0
    
    # Conversion probability (Heaviside ensures conversion only within virial radius)
    P = (2 * np.pi * const_prefact / kappa * 
         (drho_drs_converted)**-1 *  
         np.heaviside(rvirs - r_convs, 0.5) / 
         (1 + zs[:, None]))
        
    def approx_ul0(l, theta_max):
        mu = -0.5
        arg = 1- theta_max**2/2
        return  np.pi * np.sqrt((2*l+1))/2 * theta_max**1.5 * (4-theta_max**2)**0.25 * 1/sp.gamma(1-mu) * ((1+arg)/(1-arg))**(mu/2) * sp.hyp2f1(-l, l+1,1-mu, (1-arg)/2)

    
    # ========================================================================
    # Auto Power Spectrum Calculations
    # ========================================================================
    
    def Cl_1halo_pirvu(l, theta_max):
        return 4 * np.pi / (2 * l + 1) * simpson(simpson(chis[:, None]**2  * approx_ul0(l, theta_max)**2 * P**2 * n, x=ms, axis=1), x=chis, axis=0)

    def Cl_2halo(l, theta_max):
        inner_int = simpson(n[None, :, :,  None,] 
                                * n[:, None, None, :,]
                                * bias[None, :, :,  None,] 
                                * bias[:, None, None, :,]
                                * P[None, :, :, None] 
                                * P[:, None, None, :]
                                * approx_ul0(l, theta_max)[None, :, :, None]
                                * approx_ul0(l, theta_max)[:, None, None, :]
                                ,
                            x=ms,
                            axis=-1,
                            )
        m_ints = simpson(inner_int,
                        x=ms,
                        axis=-1,
                        )
        z_ints = simpson(simpson(m_ints 
                                #  * Cl_lin(l)[..., None]
                                * hls[l==ls][0] * 4 * np.pi / (2 * l + 1)
                                * chis[:, None,]**2
                                * chis[None, :,]**2,
                                x=chis,
                                axis=-1
                                ),
                        x=chis,
                        axis=-1
                        )    
        
        return 4 * np.pi / (2 * l + 1) * z_ints
    
    # ========================================================================
    # Cross-Correlation Functions  
    # ========================================================================
    
    def CgP_1h(l):
        return np.sqrt(4 * np.pi / (2 * l + 1)) * simpson(
            chis**2  * simpson(n * ug(Ws, Ncs, Nss, l=l) * P * approx_ul0(l, theta_max_array), 
                                x=ms, 
                                axis=1),
            x=chis,
            axis=0,
        )

    def CgP_2h_limber(l):
        m_ints = simpson(P * approx_ul0(l, theta_max_array) * n * bias * simpson(n
                            * bias
                            * ug(Ws, Ncs, Nss, l=l)
                            , 
                            x=ms,
                            axis=1)[:, None], 
                            x=ms, 
                            axis=1)
        Plin_at_zs = np.array([interp_lin_power((l+0.5)/chi)[chis == chi] for chi in chis]).reshape(zs.shape)
        return np.sqrt(4 * np.pi / (2 * l + 1)) * simpson(chis**2 * m_ints * Plin_at_zs, 
                                                        x=chis, 
                                                        axis=0
                                                        )

    
    # ========================================================================
    # Compute and Save Results
    # ========================================================================
    
    auto_Cls = np.array([Cl_1halo_pirvu(l, theta_max_array) + Cl_2halo(l, theta_max_array) 
                        for l in tqdm(ls, desc="Auto-power")])
    
    Cl_gP_1hs = np.array([CgP_1h(l) for l in tqdm(ls, desc="Cross 1-halo")])
    
    Cl_gP_2hs = np.array([CgP_2h_limber(l) for l in tqdm(ls, desc="Cross 2-halo")])
    
    # Save results
    return auto_Cls, Cl_gP_1hs + Cl_gP_2hs


In [ ]:
# ============================================================================
# Run Calculations for Multiple Dark Photon Masses
# ============================================================================
mA_list = [5e-13, 2e-12]  # eV
auto_Cls = []
cross_Cls = []
for mA in tqdm(mA_list, desc="Dark photon masses"):
        auto_Cl, cross_Cl = get_Cls(mA)
        auto_Cls.append(auto_Cl)
        cross_Cls.append(cross_Cl)


auto_Cls = np.array(auto_Cls)
cross_Cls = np.array(cross_Cls)

In [ ]:
plt.plot(ls, Cl_gg)
plt.xscale("log")
plt.yscale("log")
plt.xlabel(r"$\ell$")
plt.ylabel(r"$C_\ell^{\rm gg}$ ")
plt.savefig("plots/halo_auto_gal_Cls.pdf", bbox_inches="tight")


In [ ]:
# load in ILC stuff
binning = nmt.NmtBin.from_nside_linear(2048, 10)
mccarthy_Cls = np.load("/usr3/graduate/ebaker/dark_photon_constraints/ilc/pyilc_files/mccarthy_Cls.npy")
planck_Cls = np.load("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_Planck/clean_auto_Cls_binning10.npy")
mock_planck_Cls = np.load("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_Planck/clean_auto_Cls_binning10.npy")
small_beam = np.load("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_planck_small_beam/clean_auto_Cls_binning10.npy")
simple_Planck_dust = np.load("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_Planck_simple_dust/clean_auto_Cls_binning10.npy")
planck_no_pt_src = np.load("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_Planck_no_pt_srcs/clean_auto_Cls_binning10.npy")
radio = np.load("/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_alt_pt_src_100mJy/clean_auto_Cls_binning10.npy")

#normalization from McCarthy et al. 
TCMB_eV = 0.000235264 
x = 2 * np.pi * 0.000232349 /TCMB_eV
prefac = TCMB_eV * x**2 / (1- np.exp(-x))



In [ ]:
Tgamma0 = 2.73e3 # mK
epsilon = 1e-6
omega0 = 2 * np.pi * 410e6 * 6.58212e-16 # Hz to eV
omegaCMB = 2 * np.pi * 353e9 * 6.58212e-16 # Hz to eV
for i, mA in enumerate(mA_list):
    if i in [0]:
        plt.plot(ls, Tgamma0**2 * epsilon**4 / (1)**2  * auto_Cls[i], color=cols_default[i//2], label=rf"${mA:.0e}".replace("e-", r" \times 10^{-") + "}$ eV")
        # plt.plot(ls, Tgamma0**2 * epsilon**4 / omegaCMB**2  * auto_Cls[i], ls="--", color=cols_default[i//2])


# plt.plot(binning.get_effective_ells(), mccarthy_Cls[0]* prefac**2/1000, linestyle=":", label="McCarthy",  color=cols_default[0])
# plt.plot(binning.get_effective_ells(), mock_planck_Cls * prefac**2, linestyle=":", label="Mock Planck", color=cols_default[2])
# plt.plot(binning.get_effective_ells(), radio, linestyle="-", label="Radio 100mJy", color=cols_default[5])
# plt.plot(binning.get_effective_ells(), small_beam, linestyle="-", label="Planck Small Beam", color=cols_default[3])

plt.xscale("log")
plt.yscale("log")
plt.xlabel(r"$\ell$")
plt.ylabel(r"$C_\ell^{\rm A, a}$ [mK$^2$]")
auto_line = Line2D([0,1],[0,1],linestyle='-', color='k')
cross_line = Line2D([0,1],[0,1],linestyle='--', color='k')
legend2 = plt.legend([auto_line, cross_line], [r'$\omega_0=410$ MHz', r'$\omega_0=353$ GHz'], fontsize=12, loc="lower left", frameon=True)
plt.gca().add_artist(legend2)
# legend1 = plt.legend(fontsize=12, loc='center left', bbox_to_anchor=(1, 0.5))
plt.legend(fontsize=12, frameon=True)
plt.savefig("plots/halo_auto_Cls.pdf", bbox_inches="tight")


In [ ]:
for i, mA in enumerate(mA_list):
    if i in [0,2]:
        plt.plot(ls, Tgamma0 * epsilon**2 / omega0 * cross_Cls[i], color=cols_default[i//2],label=rf"${mA:.0e}".replace("e-", r" \times 10^{-") + "}$ eV")
        plt.plot(ls, Tgamma0 * epsilon**2 / omegaCMB * cross_Cls[i], ls="--", color=cols_default[i//2],)

plt.xscale("log")
plt.yscale("log")
plt.xlabel(r"$\ell$")
plt.ylabel(r"$C_\ell^{\rm A, c}$ [mK]")
auto_line = Line2D([0,1],[0,1],linestyle='-', color='k')
cross_line = Line2D([0,1],[0,1],linestyle='--', color='k')
legend2 = plt.legend([auto_line, cross_line], [r'$\omega_0=410$ MHz', r'$\omega_0=353$ GHz'], fontsize=12, loc="lower left", frameon=True)
plt.gca().add_artist(legend2)
plt.legend(fontsize=12, frameon=True)
plt.savefig("plots/halo_cross_Cls.pdf", bbox_inches="tight")